In [2]:
import pandas as pd
import numpy as np
import json
import os

EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
OUT_DIR = r"C:\Users\user\Downloads\GSE148812_clean"
AA_META = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"

# reload shortlist
shortlist_pruned_ea = pd.read_csv(os.path.join(OUT_DIR, "v2_shortlist_ld_pruned.csv"))
print("Shortlist size:", len(shortlist_pruned_ea))

# reload Y
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
encoded_df_tmp = pd.read_csv(EA_GENO, nrows=0)
sample_ids = encoded_df_tmp.columns[1:].tolist()
meta_df = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()
Y = (meta_df["smoking_status"] == "Smoker").astype(np.float64).values
print("Y shape:", Y.shape)

# build PC input
pruned_ids_ea = shortlist_pruned_ea["probe_id"].tolist()
col_names_ea = pruned_ids_ea + ["smoking_status"]

encoded_df_ea2 = pd.read_csv(EA_GENO)
probe_rows_final = encoded_df_ea2[encoded_df_ea2["probe_id"].isin(set(pruned_ids_ea))].copy()
probe_rows_final = probe_rows_final.set_index("probe_id").reindex(pruned_ids_ea)
X_pc_ea = probe_rows_final.to_numpy(dtype=np.float64).T

X_pc_full_ea = np.hstack([X_pc_ea, Y.reshape(-1, 1)])
print("PC input shape:", X_pc_full_ea.shape)

np.save(os.path.join(OUT_DIR, "v2_pc_input.npy"), X_pc_full_ea)
with open(os.path.join(OUT_DIR, "v2_pc_col_names.json"), "w") as f:
    json.dump(col_names_ea, f)
print("Saved.")

Shortlist size: 19
Y shape: (1595,)
PC input shape: (1595, 20)
Saved.


In [4]:
import numpy as np
import pandas as pd
import os

OUT_DIR = r"C:\Users\user\Downloads\GSE148812_clean"
EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"

shortlist_pruned_ea = pd.read_csv(os.path.join(OUT_DIR, "v2_shortlist_ld_pruned.csv"))
pruned_ids_ea = shortlist_pruned_ea["probe_id"].tolist()

encoded_df_ea = pd.read_csv(EA_GENO)
probe_rows = encoded_df_ea[encoded_df_ea["probe_id"].isin(set(pruned_ids_ea))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(pruned_ids_ea)
X_check = probe_rows.to_numpy(dtype=np.float64).T  # samples x SNPs

# compute full correlation matrix
corr_matrix = np.corrcoef(X_check.T)

# find pairs with |r| > 0.99 (near-perfect correlation)
perfect_pairs = []
for i in range(len(pruned_ids_ea)):
    for j in range(i+1, len(pruned_ids_ea)):
        if abs(corr_matrix[i,j]) > 0.99:
            perfect_pairs.append((pruned_ids_ea[i], pruned_ids_ea[j], corr_matrix[i,j]))

print(f"Near-perfect correlation pairs (|r|>0.99): {len(perfect_pairs)}")
for p1, p2, r in perfect_pairs:
    print(f"  r={r:.6f}: {p1[:30]} | {p2[:30]}")

# remove one from each perfect pair (keep higher stability)
to_remove = set()
for p1, p2, r in perfect_pairs:
    stab1 = shortlist_pruned_ea.loc[shortlist_pruned_ea["probe_id"]==p1, "stability_fraction"].values[0]
    stab2 = shortlist_pruned_ea.loc[shortlist_pruned_ea["probe_id"]==p2, "stability_fraction"].values[0]
    to_remove.add(p2 if stab1 >= stab2 else p1)

print(f"\nRemoving {len(to_remove)} perfectly correlated SNPs")
shortlist_final_ea = shortlist_pruned_ea[~shortlist_pruned_ea["probe_id"].isin(to_remove)].copy()
print(f"Final shortlist: {len(shortlist_final_ea)} SNPs")

shortlist_final_ea.to_csv(os.path.join(OUT_DIR, "v2_shortlist_final.csv"), index=False)
print("Saved.")

Near-perfect correlation pairs (|r|>0.99): 1
  r=1.000000: exm1215705-0_T_F_1921777411 | exm244139-0_T_R_1918920506

Removing 1 perfectly correlated SNPs
Final shortlist: 18 SNPs
Saved.


In [1]:
import json
import numpy as np
import pandas as pd
import os

OUT_DIR = r"C:\Users\user\Downloads\GSE148812_clean"
EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
AA_META = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"

shortlist_final_ea = pd.read_csv(os.path.join(OUT_DIR, "v2_shortlist_final.csv"))
pruned_ids_ea = shortlist_final_ea["probe_id"].tolist()
col_names_ea = pruned_ids_ea + ["smoking_status"]

encoded_df_ea = pd.read_csv(EA_GENO)
sample_ids = encoded_df_ea.columns[1:].tolist()

meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()
Y = (meta_df["smoking_status"] == "Smoker").astype(np.float64).values

probe_rows = encoded_df_ea[encoded_df_ea["probe_id"].isin(set(pruned_ids_ea))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(pruned_ids_ea)
X_pc_ea = probe_rows.to_numpy(dtype=np.float64).T

X_pc_full_ea = np.hstack([X_pc_ea, Y.reshape(-1, 1)])
print("PC input shape:", X_pc_full_ea.shape)  # expect (1595, 19)

np.save(os.path.join(OUT_DIR, "v2_pc_input.npy"), X_pc_full_ea)
with open(os.path.join(OUT_DIR, "v2_pc_col_names.json"), "w") as f:
    json.dump(col_names_ea, f)
print("Saved.")

PC input shape: (1595, 19)
Saved.
